In [ ]:
# CELL 1 — Install & Mount Drive
# Run once per session
# ════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

# Install any missing packages
# !pip install transformers datasets scikit-learn -q

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 2 — Imports
# Run once per session
# ════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gc
import re
import zipfile
import os

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score

print("✅ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")



In [ ]:
# CELL 3 — GPU Memory Check Helper
# Run anytime to see memory status
# ════════════════════════════════════════════════════════════════════

def check_gpu():
    if torch.cuda.is_available():
        total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved  = torch.cuda.memory_reserved() / 1024**3
        free      = total - allocated
        print(f"GPU : {torch.cuda.get_device_name(0)}")
        print(f"Total    : {total:.1f} GB")
        print(f"Allocated: {allocated:.2f} GB")
        print(f"Reserved : {reserved:.2f} GB")
        print(f"Free     : {free:.2f} GB")
    else:
        print("⚠️ No GPU available — check Runtime > Change runtime type > T4 GPU")

check_gpu()


In [ ]:
# CELL 4 — Cleanup Helper  ⚠️ RUN BETWEEN EVERY EXPERIMENT
# ════════════════════════════════════════════════════════════════════

def cleanup(vars_to_delete=None):
    """
    Call this between experiments to free GPU memory.
    Usage:
        cleanup()                          # cleans common names
        cleanup(['model', 'trainer'])      # cleans specific names
    """
    default_names = ['model', 'trainer', 'train_tokenized', 'dev_tokenized']
    names = vars_to_delete if vars_to_delete else default_names

    import builtins
    g = globals()
    for name in names:
        if name in g:
            del g[name]
            print(f"  deleted: {name}")

    torch.cuda.empty_cache()
    gc.collect()
    print("✅ Memory cleared")
    check_gpu()

# Example usage (don't run now, run between experiments):
# cleanup()

In [ ]:
TRAIN_PATH = "/content/drive/MyDrive/trainData.csv"
DEV_PATH   = "/content/drive/MyDrive/devData.csv"
OUTPUT_DIR = "/content/drive/MyDrive/smm4h_task1"
TRAIN_CADEC_PATH="/content/drive/MyDrive/trainCadecData.csv"
DEV_CADEC_PATH="/content/drive/MyDrive/devCadecData.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

df_train = pd.read_csv(TRAIN_PATH)
df_dev   = pd.read_csv(DEV_PATH)
df_train_cadec=pd.read_csv(TRAIN_CADEC_PATH)
df_cadec_dev=pd.read_csv(DEV_CADEC_PATH)

In [ ]:
# CELL 5 — Load Raw Data
# Run once per session
# ════════════════════════════════════════════════════════════════════

# ── UPDATE THESE PATHS to match your Google Drive ──


# df_dev_original=pd.read_csv(DEV_PATH)

df_train=pd.concat([df_train,df_train_cadec])
df_dev=pd.concat([df_dev,df_cadec_dev])

df_train['label'] = df_train['label'].astype(int)
df_dev['label']   = df_dev['label'].astype(int)
# df_dev_original['label']=df_dev_original['label'].astype(int)

print("✅ Data loaded")
print(f"Train: {df_train.shape} | Dev: {df_dev.shape}")

# New Section

In [ ]:

# ════════════════════════════════════════════════════════════════════
# CELL 6 — Data Inspection
# Run once to understand your data
# ════════════════════════════════════════════════════════════════════

print("=" * 55)
print("COLUMNS:", df_train.columns.tolist())

print("\n📊 Label Distribution (Train):")
print(df_train.groupby('language')['label'].value_counts().unstack(fill_value=0))

print("\n📏 Text Length Stats by Language (Train):")
df_train['text_len'] = df_train['text'].str.len()
print(df_train.groupby('language')['text_len'].describe()[['min','mean','max']].round(1))

print("\n⚠️ Missing Values:")
print(df_train.isnull().sum())

print("\n⚠️ Duplicate Texts:", df_train.duplicated(subset=['text']).sum())
print("⚠️ Empty Texts:", (df_train['text'].str.strip().str.len() == 0).sum())

print("\n📝 Sample ADE Post (label=1):")
print(df_train[df_train['label']==1]['text'].iloc[0])

print("\n📝 Sample Non-ADE Post (label=0):")
print(df_train[df_train['label']==0]['text'].iloc[0])



In [ ]:

# # ════════════════════════════════════════════════════════════════════
# # CELL 7 — Data Cleaning
# # Run once after inspection
# # ════════════════════════════════════════════════════════════════════

# def clean_text(text):
#     if not isinstance(text, str):
#         return ""
#     # Fix HTML encoding artifacts
#     text = text.replace('&amp;', '&')
#     text = text.replace('&lt;', '<')
#     text = text.replace('&gt;', '>')
#     text = text.replace('&quot;', '"')
#     text = text.replace('&#39;', "'")
#     text = text.replace('\x00', '')       # null characters
#     # Remove URLs (no signal for ADE detection)
#     text = re.sub(r'http\S+|www\S+', '', text)
#     # Remove excessive whitespace
#     text = re.sub(r'\s+', ' ', text).strip()
#     return text

# def clean_dataframe(df, name=""):
#     original_len = len(df)

#     # 1. Fix text content
#     df['text'] = df['text'].apply(clean_text)

#     # 2. Drop null text rows
#     df = df.dropna(subset=['text'])

#     # 3. Drop empty text rows
#     df = df[df['text'].str.strip().str.len() > 0]

#     # 4. Remove duplicate (text + label) pairs
#     df = df.drop_duplicates(subset=['text', 'label'])

#     # 5. Remove conflicting labels (same text, different label)
#     conflicts = df.groupby('text')['label'].nunique()
#     conflict_texts = conflicts[conflicts > 1].index
#     if len(conflict_texts) > 0:
#         print(f"  ⚠️ Removing {len(conflict_texts)} conflicting texts")
#         df = df[~df['text'].isin(conflict_texts)]

#     removed = original_len - len(df)
#     print(f"{name}: {original_len} → {len(df)} rows (removed {removed})")
#     return df.reset_index(drop=True)

# df_train = clean_dataframe(df_train, "Train")

# # Drop helper column
# if 'text_len' in df_train.columns:
#     df_train = df_train.drop(columns=['text_len'])

# # Sanity checks
# assert df_train['label'].isnull().sum() == 0, "NaN labels in train!"
# assert df_dev['label'].isnull().sum() == 0,   "NaN labels in dev!"
# assert df_train['text'].isnull().sum() == 0,  "NaN texts in train!"
# assert df_train['label'].isin([0,1]).all(),    "Invalid labels!"
# print("✅ All sanity checks passed — data is clean!")

In [ ]:

# ════════════════════════════════════════════════════════════════════
# CELL 8 — Define Metrics & Weighted Trainer
# Run once per session (these are reusable across experiments)
# ════════════════════════════════════════════════════════════════════

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    return {
        "macro_f1" : f1_score(labels, predictions, average='macro'),
        "f1_pos"   : f1_score(labels, predictions, pos_label=1, average='binary'),
    }

# Computed here; updated per experiment if class balance changes
labels_array  = df_train['label'].values
class_weights = compute_class_weight('balanced', classes=np.array([0,1]), y=labels_array)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print(f"Class weights → 0: {class_weights[0]:.3f}  |  1: {class_weights[1]:.3f}")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("✅ WeightedTrainer and compute_metrics defined")



In [ ]:
# CELL 9 — Tokenize
# 🔁 RE-RUN FOR EACH EXPERIMENT (when changing MODEL_NAME or max_length)
# ════════════════════════════════════════════════════════════════════

# ── CHANGE THIS per experiment ──
MODEL_NAME = "xlm-roberta-large"   # or "xlm-roberta-large"
MAX_LENGTH = 128                   # try 256 in later experiment
MODEL_PATH_LARGE='/content/drive/MyDrive/smm4h_task1/model_exp1_large_128'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True
    )

train_dataset = Dataset.from_pandas( df_train[['text', 'label']] )
dev_dataset   = Dataset.from_pandas( df_dev[['text', 'label']] )
# dev_dataset_original=Dataset.from_pandas( df_dev_original[['text', 'label']] )

train_tokenized = train_dataset.map(tokenize, batched=True)
dev_tokenized   = dev_dataset.map(tokenize, batched=True)
# dev_original_tokenized=dev_dataset_original.map(tokenize, batched=True)
data_collator   = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"✅ Tokenized with xlm-roberta-large | max_length={MAX_LENGTH}")
print(f"   Train: {len(train_tokenized)} | Dev: {len(dev_tokenized)}")

In [ ]:
# print(df_dev.shape)
# print(dev_tokenized.shape)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# CELL 10 — Train
# 🔁 RE-RUN FOR EACH EXPERIMENT
# ════════════════════════════════════════════════════════════════════

check_gpu()  # check memory before loading model

# ── Adjust batch size based on model ──
# xlm-roberta-base  → batch=16, accum=2  (effective 32)
# xlm-roberta-large → batch=8,  accum=4  (effective 32)
BATCH_SIZE   = 8
ACCUM_STEPS  = 4
LR           = 1e-5    # use 1e-5 for xlm-roberta-large
EPOCHS       = 3
EXPERIMENT   = "exp1_large_128"   # change name for each experiment



model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

training_args = TrainingArguments(
    output_dir                  = f"{OUTPUT_DIR}/results_{EXPERIMENT}",
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = ACCUM_STEPS,
    per_device_eval_batch_size  = 32,
    learning_rate               = LR,
    weight_decay                = 0.01,
    warmup_steps                = 200,
    lr_scheduler_type           = "cosine",
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "macro_f1",
    greater_is_better           = True,
    save_total_limit            = 2,
    fp16                        = True,
    seed                        = 42,
    data_seed                   = 42,
    report_to                   = "none",
)

trainer = WeightedTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_tokenized,
    eval_dataset     = dev_tokenized,
    compute_metrics  = compute_metrics,
    processing_class = tokenizer,
    data_collator    = data_collator,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

In [ ]:
# CELL 11 — Per-Language Evaluation
# Run after each training
# ════════════════════════════════════════════════════════════════════


preds_output = trainer.predict(dev_tokenized)
probs= F.softmax(torch.tensor(preds_output.predictions), dim=-1)[:, 1].numpy()

pred_labels  = np.argmax(preds_output.predictions, axis=-1)
# probs        = F.softmax(torch.tensor(preds_output.predictions), dim=-1)[:, 1].numpy()

df_eval         = df_dev.copy() # Changed from dev_tokenized.copy()
df_eval['pred'] = pred_labels
df_eval['prob'] = probs


print("=" * 55)
print("📊 Per-Language Results on Dev Set")
print("=" * 55)
for lang in sorted(df_eval['language'].unique()):
    subset  = df_eval[df_eval['language'] == lang]
    macro   = f1_score(subset['label'], subset['pred'], average='macro')
    pos_f1  = f1_score(subset['label'], subset['pred'], pos_label=1, average='binary')
    support = subset['label'].sum()
    print(f"  {lang:4s} | macro F1: {macro:.4f} | pos F1: {pos_f1:.4f} | ADE count: {support}")

overall = f1_score(df_eval['label'], df_eval['pred'], average='macro')
print(f"\n  {'OVERALL':4s} | macro F1: {overall:.4f}")

In [ ]:
print(df_dev.shape)
print(probs.shape)
print(dev_tokenized.shape)

In [ ]:


# ════════════════════════════════════════════════════════════════════
# CELL 12 — Threshold Tuning
# Run after evaluation — find the best classification threshold
# ════════════════════════════════════════════════════════════════════

print("🌍 Global Threshold Search:")
print("-" * 40)
best_f1, best_threshold = 0, 0.5

for t in [i/100 for i in range(20, 70)]:
    preds_t = (probs >= t).astype(int)
    f1      = f1_score(df_eval['label'], preds_t, average='macro')
    marker  = " ← BEST" if f1 > best_f1 else ""
    if f1 > best_f1:
        best_f1        = f1
        best_threshold = t
    print(f"  t={t:.2f}  →  macro F1: {f1:.4f}{marker}")

print(f"\n✅ Best threshold : {best_threshold:.2f}")
print(f"✅ Best macro F1  : {best_f1:.4f}")
print(f"   (vs default 0.50: {f1_score(df_eval['label'], pred_labels, average='macro'):.4f})")


In [ ]:

# ════════════════════════════════════════════════════════════════════
# CELL 13 — Save Predictions & Model
# Run after evaluation and threshold tuning
# ════════════════════════════════════════════════════════════════════

# Apply best threshold
final_preds = (probs >= best_threshold).astype(int)

# Save submission CSV
submission_path = f"{OUTPUT_DIR}/predictions_{EXPERIMENT}.csv"
submission      = df_dev[['id']].copy()
submission['predicted_label'] = final_preds
submission.to_csv(submission_path, index=False)

# Verify submission
print("📋 Submission preview:")
print(submission.head())
print(f"\nLabel distribution: {submission['predicted_label'].value_counts().to_dict()}")
print(f"Total rows: {len(submission)} (dev rows: {len(df_dev)})")
assert len(submission) == len(df_dev), "❌ Row count mismatch!"
assert submission['predicted_label'].isin([0,1]).all(), "❌ Invalid labels!"
print("✅ Submission verified")

# Save trained model to Drive
model_save_path = f"{OUTPUT_DIR}/model_{EXPERIMENT}"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"✅ Model saved to: {model_save_path}")

# Create ZIP for CodaBench
zip_path = f"{OUTPUT_DIR}/submit_{EXPERIMENT}.zip"
with zipfile.ZipFile(zip_path, 'w') as zf:
    zf.write(submission_path, arcname="predictions_dev.csv")
print(f"✅ ZIP created: {zip_path}")
print(f"\n→ Download {zip_path} from Drive and upload to CodaBench")



In [ ]:
for index, row in df_train.iterrows():
  if row['id']=='en_235':
    print("found")


ar=['en_235', 'en_305', 'en_273', 'en_374', 'en_297', 'en_906', 'en_837', 'en_870', 'en_26', 'ru_2892', 'en_479', 'en_155', 'en_106', 'en_625', 'fr_319', 'en_300']
# print(ar)
print(len(ar))

cnt=0

for it in ar:
  for index, row in df_train_cadec.iterrows():
    if row['id']==it:
      cnt=cnt+1

print(cnt)